# 06. LightGBM 위험도 및 리드타임 모델

이 노트북은 `04_feature_engineering` 산출물과 `05_baseline_anomaly_model` 산출물을 결합해 Agent 전달 전 단계의 supervised ML 결과를 만든다.

- 06-A: LightGBM 위험도 이진 분류
- 06-B: LightGBM 리드타임 3중분류
- 06-C: Agent 전달 전 결합 산출물 저장

ML은 최종 우선순위를 직접 결정하지 않는다. 이 노트북은 `risk_probability`, `lead_time_bucket`, `confidence`, feature importance를 제공하고, 최종 판단은 Agent 단계에서 수행한다.

In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import json
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

RANDOM_STATE = 42
RUN_ID = datetime.now().strftime("run_%Y%m%d_%H%M%S")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    candidates = [PROJECT_ROOT, *PROJECT_ROOT.parents]
    PROJECT_ROOT = next(path for path in candidates if (path / "data").exists())

FEATURE_DIR = PROJECT_ROOT / "data" / "processed" / "ml_features"
ANOMALY_DIR = PROJECT_ROOT / "data" / "processed" / "ml_baseline"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "ml_supervised"
RUN_DIR = OUTPUT_DIR / "runs" / RUN_ID
MODEL_DIR = RUN_DIR / "models"
PLOT_DIR = RUN_DIR / "plots"

for path in [OUTPUT_DIR, RUN_DIR, MODEL_DIR, PLOT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TRAINABLE_PATH = FEATURE_DIR / "trainable_windows.csv"
FEATURE_COLUMNS_PATH = FEATURE_DIR / "feature_columns.csv"
ANOMALY_SCORES_PATH = ANOMALY_DIR / "anomaly_baseline_scores.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RUN_ID:", RUN_ID)

## 1. 입력 데이터 로드

06은 04에서 고정한 feature 후보와 05에서 생성한 `anomaly_score`를 결합한다. 결합 키는 기계실, 파일명, window 시작/종료 시각을 사용한다.

In [ ]:
required_paths = [TRAINABLE_PATH, FEATURE_COLUMNS_PATH, ANOMALY_SCORES_PATH]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("06 실행 전에 04와 05 산출물이 필요합니다: " + ", ".join(missing_paths))

windows = pd.read_csv(TRAINABLE_PATH)
feature_columns = pd.read_csv(FEATURE_COLUMNS_PATH)
anomaly_scores = pd.read_csv(ANOMALY_SCORES_PATH)

merge_keys = ["manufacturer", "substation_id", "source_file", "window_start", "window_end"]
anomaly_keep = merge_keys + [
    "anomaly_score",
    "anomaly_threshold",
    "anomaly_label",
    "anomaly_model_key",
]
anomaly_keep = [col for col in anomaly_keep if col in anomaly_scores.columns]

df = windows.merge(anomaly_scores[anomaly_keep], on=merge_keys, how="left", validate="one_to_one")

print("windows:", windows.shape)
print("anomaly_scores:", anomaly_scores.shape)
print("merged:", df.shape)
display(df[["label", "split_time_based", "use_for_supervised_training", "anomaly_score"]].head())

## 2. 학습 대상과 feature set 정의

위험도 모델은 `normal`과 `pre_fault`를 이진 분류한다. 리드타임 모델은 `pre_fault` 중 `estimated_lead_time_hours`가 있는 row만 사용하고, `0~24h`, `24~72h`, `72h+` 3개 클래스로 나눈다.

리드타임 모델에 `risk_probability`를 직접 넣으면 같은 train set에서 학습된 위험도 모델 결과가 label proxy처럼 작동할 수 있으므로, 이번 baseline에서는 04 feature와 05 anomaly 결과만 사용한다. 최종 export에서는 두 모델 결과를 나란히 결합한다.

In [ ]:
def as_bool(series: pd.Series) -> pd.Series:
    return series.astype(str).str.lower().isin(["true", "1", "yes"])

feature_name_col = "column" if "column" in feature_columns.columns else "feature"

risk_base_features = feature_columns.loc[
    as_bool(feature_columns["present_in_dataset"]) & as_bool(feature_columns["risk_feature"]),
    feature_name_col,
].tolist()
leadtime_base_features = feature_columns.loc[
    as_bool(feature_columns["present_in_dataset"]) & as_bool(feature_columns["leadtime_feature"]),
    feature_name_col,
].tolist()

model_result_features = ["anomaly_score", "anomaly_label"]
leakage_or_target_cols = {
    "label",
    "fault_label",
    "fault_event_id",
    "estimated_lead_time_hours",
    "risk_target",
    "lead_time_bucket",
    "risk_probability",
    "risk_score",
}

def available_numeric_features(candidates: list[str]) -> list[str]:
    features = []
    for col in candidates:
        if col in df.columns and col not in leakage_or_target_cols and col not in features:
            features.append(col)
    return features

risk_features = available_numeric_features(risk_base_features + model_result_features)
leadtime_features = available_numeric_features(leadtime_base_features + model_result_features)

df["risk_target"] = (df["label"] == "pre_fault").astype(int)

def make_lead_time_bucket(hours: float) -> str | float:
    if pd.isna(hours):
        return np.nan
    if hours <= 24:
        return "short_0_24h"
    if hours <= 72:
        return "mid_24_72h"
    return "long_72h_plus"

df["lead_time_bucket"] = df["estimated_lead_time_hours"].apply(make_lead_time_bucket)

supervised_mask = df["use_for_supervised_training"].astype(str).str.lower().eq("true")
risk_mask = supervised_mask & df["label"].isin(["normal", "pre_fault"])
leadtime_mask = supervised_mask & df["label"].eq("pre_fault") & df["lead_time_bucket"].notna()

print("risk feature count:", len(risk_features))
print("leadtime feature count:", len(leadtime_features))
display(df.loc[risk_mask, ["split_time_based", "label"]].value_counts().rename("rows").reset_index())
display(df.loc[leadtime_mask, ["split_time_based", "lead_time_bucket"]].value_counts().rename("rows").reset_index())

## 3. 공통 학습/평가 함수

이번 baseline은 time-based split을 기본으로 사용한다. 모델 선택은 validation 기준으로 수행하고, holdout은 최종 일반화 확인용으로만 본다.

In [ ]:
def make_xy(frame: pd.DataFrame, features: list[str], target_col: str):
    X = frame[features].apply(pd.to_numeric, errors="coerce")
    y = frame[target_col]
    return X, y

def binary_metrics(y_true, y_prob, threshold: float) -> dict:
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": threshold,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "average_precision": average_precision_score(y_true, y_prob),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "false_positive_rate": fp / (fp + tn) if (fp + tn) else np.nan,
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    }

def multiclass_metrics(y_true, y_pred) -> dict:
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

def save_latest_and_run(df_out: pd.DataFrame, latest_name: str):
    latest_path = OUTPUT_DIR / latest_name
    run_path = RUN_DIR / latest_name
    df_out.to_csv(latest_path, index=False)
    df_out.to_csv(run_path, index=False)
    return latest_path, run_path

## 4. 06-A 위험도 이진 분류

위험도 모델은 고장 확정 모델이 아니라, 고장 신고 전 위험 패턴과의 유사도를 산출하는 모델이다. 따라서 accuracy보다 `f1`, `recall`, `precision`, `false_positive_rate`, `average_precision`을 함께 본다.

In [ ]:
risk_train = df[risk_mask & df["split_time_based"].eq("train")].copy()
risk_val = df[risk_mask & df["split_time_based"].eq("validation")].copy()
risk_holdout = df[risk_mask & df["split_time_based"].eq("holdout")].copy()

X_train, y_train = make_xy(risk_train, risk_features, "risk_target")
X_val, y_val = make_xy(risk_val, risk_features, "risk_target")
X_holdout, y_holdout = make_xy(risk_holdout, risk_features, "risk_target")

risk_grid = [
    {"n_estimators": n, "learning_rate": lr, "num_leaves": leaves, "min_child_samples": child}
    for n in [300, 600]
    for lr in [0.03, 0.05]
    for leaves in [15, 31]
    for child in [20, 50]
]
risk_thresholds = np.round(np.arange(0.20, 0.951, 0.025), 3)
RISK_FP_GUARD = 0.05
RISK_PRECISION_GUARD = 0.70

risk_rows = []
risk_models = {}

for params in risk_grid:
    model_key = "|".join(f"{key}={value}" for key, value in params.items())
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            objective="binary",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )),
    ])
    pipe.fit(X_train, y_train)
    val_prob = pipe.predict_proba(X_val)[:, 1]
    risk_models[model_key] = pipe
    for threshold in risk_thresholds:
        row = binary_metrics(y_val, val_prob, threshold)
        row.update(params)
        row["model_key"] = model_key
        row["split"] = "validation"
        risk_rows.append(row)

risk_experiments_raw = pd.DataFrame(risk_rows)
guard_candidates = risk_experiments_raw[
    (risk_experiments_raw["false_positive_rate"] <= RISK_FP_GUARD)
    & (risk_experiments_raw["precision"] >= RISK_PRECISION_GUARD)
].copy()

if guard_candidates.empty:
    risk_selection_pool = risk_experiments_raw.copy()
    risk_threshold_strategy = "fallback_f1_priority"
else:
    risk_selection_pool = guard_candidates
    risk_threshold_strategy = "validation_fp_precision_guard"

risk_experiments = risk_selection_pool.sort_values(
    ["f1", "false_positive_rate", "precision", "recall", "average_precision", "roc_auc"],
    ascending=[False, True, False, False, False, False],
).reset_index(drop=True)

risk_experiment_results = risk_experiments_raw.sort_values(
    ["f1", "false_positive_rate", "precision", "recall", "average_precision", "roc_auc"],
    ascending=[False, True, False, False, False, False],
).reset_index(drop=True)

best_risk = risk_experiments.iloc[0].to_dict()
best_risk["threshold_strategy"] = risk_threshold_strategy
best_risk["false_positive_guard"] = RISK_FP_GUARD
best_risk["precision_guard"] = RISK_PRECISION_GUARD
best_risk_model = risk_models[best_risk["model_key"]]
best_risk_threshold = float(best_risk["threshold"])

holdout_prob = best_risk_model.predict_proba(X_holdout)[:, 1]
risk_holdout_metrics = binary_metrics(y_holdout, holdout_prob, best_risk_threshold)

validation_prob = best_risk_model.predict_proba(X_val)[:, 1]
risk_threshold_diagnostics = []
for threshold in risk_thresholds:
    validation_row = binary_metrics(y_val, validation_prob, float(threshold))
    validation_row.update({"run_id": RUN_ID, "model_key": best_risk["model_key"], "split": "validation"})
    holdout_row = binary_metrics(y_holdout, holdout_prob, float(threshold))
    holdout_row.update({"run_id": RUN_ID, "model_key": best_risk["model_key"], "split": "holdout"})
    risk_threshold_diagnostics.extend([validation_row, holdout_row])
risk_threshold_diagnostics = pd.DataFrame(risk_threshold_diagnostics)

def risk_group_diagnostics(frame: pd.DataFrame, probabilities: np.ndarray, split_name: str) -> pd.DataFrame:
    diag = frame[["manufacturer", "configuration_type", "season_bucket", "split_regime_based", "risk_target"]].copy()
    diag["risk_probability"] = probabilities
    diag["risk_pred"] = (diag["risk_probability"] >= best_risk_threshold).astype(int)
    rows = []
    for group_col in ["manufacturer", "configuration_type", "season_bucket", "split_regime_based"]:
        for group_value, group in diag.groupby(group_col, dropna=False):
            if group["risk_target"].nunique() < 2:
                continue
            row = binary_metrics(group["risk_target"], group["risk_probability"], best_risk_threshold)
            row.update({"run_id": RUN_ID, "split": split_name, "group_column": group_col, "group_value": group_value, "rows": len(group)})
            rows.append(row)
    return pd.DataFrame(rows)

risk_group_metrics = pd.concat([
    risk_group_diagnostics(risk_val, validation_prob, "validation"),
    risk_group_diagnostics(risk_holdout, holdout_prob, "holdout"),
], ignore_index=True)

print("risk threshold strategy:", risk_threshold_strategy)
display(risk_experiments.head(20))
display(risk_threshold_diagnostics.sort_values(["split", "false_positive_rate", "f1"], ascending=[True, True, False]).head(20))
display(risk_group_metrics.sort_values(["split", "false_positive_rate"], ascending=[True, False]).head(20))
print("best risk:", best_risk)
print("risk holdout:", risk_holdout_metrics)

## 5. 06-B 리드타임 3중분류

리드타임은 위험 후보 구간에서 의미가 있으므로 `pre_fault` 샘플만 사용한다. 이번 baseline의 구간은 다음과 같다.

- `short_0_24h`: 24시간 이내
- `mid_24_72h`: 24시간 초과 72시간 이내
- `long_72h_plus`: 72시간 초과

In [ ]:
lead_train = df[leadtime_mask & df["split_time_based"].eq("train")].copy()
lead_val = df[leadtime_mask & df["split_time_based"].eq("validation")].copy()
lead_holdout = df[leadtime_mask & df["split_time_based"].eq("holdout")].copy()

X_lead_train, y_lead_train = make_xy(lead_train, leadtime_features, "lead_time_bucket")
X_lead_val, y_lead_val = make_xy(lead_val, leadtime_features, "lead_time_bucket")
X_lead_holdout, y_lead_holdout = make_xy(lead_holdout, leadtime_features, "lead_time_bucket")

lead_grid = [
    {"n_estimators": n, "learning_rate": lr, "num_leaves": leaves, "min_child_samples": child}
    for n in [300, 600]
    for lr in [0.03, 0.05]
    for leaves in [15, 31]
    for child in [10, 30]
]

lead_rows = []
lead_models = {}

for params in lead_grid:
    model_key = "|".join(f"{key}={value}" for key, value in params.items())
    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", LGBMClassifier(
            objective="multiclass",
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
            **params,
        )),
    ])
    pipe.fit(X_lead_train, y_lead_train)
    val_pred = pipe.predict(X_lead_val)
    row = multiclass_metrics(y_lead_val, val_pred)
    row.update(params)
    row["model_key"] = model_key
    row["split"] = "validation"
    lead_rows.append(row)
    lead_models[model_key] = pipe

lead_experiments = pd.DataFrame(lead_rows).sort_values(
    ["macro_f1", "weighted_f1", "macro_recall", "accuracy"],
    ascending=[False, False, False, False],
).reset_index(drop=True)

best_lead = lead_experiments.iloc[0].to_dict()
best_lead_model = lead_models[best_lead["model_key"]]
lead_holdout_pred = best_lead_model.predict(X_lead_holdout)
lead_holdout_metrics = multiclass_metrics(y_lead_holdout, lead_holdout_pred)

display(lead_experiments.head(20))
print("best leadtime:", best_lead)
print("leadtime holdout:", lead_holdout_metrics)
print(classification_report(y_lead_holdout, lead_holdout_pred, zero_division=0))

## 6. 예측 결과 및 Agent 전 단계 산출물 생성

모든 window에 대해 위험도 확률과 리드타임 예측을 생성한다. 리드타임 예측은 모델상 모든 row에 대해 계산할 수 있지만, 해석상 `risk_class == 1` 또는 위험도가 높은 row에서 의미가 크다.

In [ ]:
X_all_risk = df[risk_features].apply(pd.to_numeric, errors="coerce")
risk_probability = best_risk_model.predict_proba(X_all_risk)[:, 1]
risk_class = (risk_probability >= best_risk_threshold).astype(int)

medium_cutoff = best_risk_threshold * 0.5
risk_level = np.select(
    [risk_probability >= best_risk_threshold, risk_probability >= medium_cutoff],
    ["high", "medium"],
    default="low",
)

X_all_lead = df[leadtime_features].apply(pd.to_numeric, errors="coerce")
lead_pred = best_lead_model.predict(X_all_lead)
lead_prob = best_lead_model.predict_proba(X_all_lead)
lead_classes = list(best_lead_model.named_steps["model"].classes_)
lead_confidence = lead_prob.max(axis=1)

prediction_cols = [
    "manufacturer",
    "substation_id",
    "source_file",
    "window_start",
    "window_end",
    "label",
    "fault_label",
    "fault_event_id",
    "estimated_lead_time_hours",
    "split_time_based",
    "split_regime_based",
    "configuration_type",
    "season_bucket",
    "anomaly_score",
    "anomaly_threshold",
    "anomaly_label",
]
prediction_cols = [col for col in prediction_cols if col in df.columns]
combined_outputs = df[prediction_cols].copy()
combined_outputs["risk_probability"] = risk_probability
combined_outputs["risk_score"] = risk_probability
combined_outputs["risk_threshold"] = best_risk_threshold
combined_outputs["risk_class"] = risk_class
combined_outputs["risk_level"] = risk_level
combined_outputs["lead_time_bucket"] = lead_pred
combined_outputs["lead_time_confidence"] = lead_confidence
combined_outputs["risk_model_version"] = "risk_lgbm_06_hsj_v1"
combined_outputs["leadtime_model_version"] = "leadtime_lgbm_06_hsj_v1"
combined_outputs["run_id"] = RUN_ID

for idx_class, class_name in enumerate(lead_classes):
    combined_outputs[f"leadtime_prob_{class_name}"] = lead_prob[:, idx_class]

risk_predictions = combined_outputs[prediction_cols + [
    "risk_probability", "risk_score", "risk_threshold", "risk_class", "risk_level", "risk_model_version", "run_id"
]].copy()
leadtime_predictions = combined_outputs[prediction_cols + [
    "risk_probability", "risk_class", "lead_time_bucket", "lead_time_confidence", "leadtime_model_version", "run_id"
] + [f"leadtime_prob_{class_name}" for class_name in lead_classes]].copy()

display(combined_outputs.head())

## 7. 성능 지표, feature importance, 실행 이력 저장

최신본 파일과 run별 파일을 모두 저장한다. 최신본은 다음 노트북이 읽기 쉽도록 고정 이름으로 유지하고, run 폴더는 실험 추적용으로 둔다.

In [ ]:
risk_metrics = pd.DataFrame([
    {"run_id": RUN_ID, "model": "risk_lgbm", "split": "validation", **{k: best_risk[k] for k in best_risk if k not in ["split"]}},
    {"run_id": RUN_ID, "model": "risk_lgbm", "split": "holdout", **risk_holdout_metrics, "model_key": best_risk["model_key"]},
])
lead_metrics = pd.DataFrame([
    {"run_id": RUN_ID, "model": "leadtime_lgbm", "split": "validation", **{k: best_lead[k] for k in best_lead if k not in ["split"]}},
    {"run_id": RUN_ID, "model": "leadtime_lgbm", "split": "holdout", **lead_holdout_metrics, "model_key": best_lead["model_key"]},
])
model_metrics = pd.concat([risk_metrics, lead_metrics], ignore_index=True, sort=False)

risk_importance = pd.DataFrame({
    "feature": risk_features,
    "importance": best_risk_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)
risk_importance["model"] = "risk_lgbm"

lead_importance = pd.DataFrame({
    "feature": leadtime_features,
    "importance": best_lead_model.named_steps["model"].feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)
lead_importance["model"] = "leadtime_lgbm"

feature_importance = pd.concat([risk_importance, lead_importance], ignore_index=True)

save_latest_and_run(risk_experiment_results, "risk_experiment_results.csv")
save_latest_and_run(risk_experiments, "risk_selected_candidate_pool.csv")
save_latest_and_run(risk_threshold_diagnostics, "risk_threshold_diagnostics.csv")
save_latest_and_run(risk_group_metrics, "risk_group_diagnostics.csv")
save_latest_and_run(lead_experiments, "leadtime_experiment_results.csv")
save_latest_and_run(model_metrics, "risk_leadtime_model_metrics.csv")
save_latest_and_run(risk_predictions, "risk_predictions.csv")
save_latest_and_run(leadtime_predictions, "leadtime_predictions.csv")
save_latest_and_run(combined_outputs, "agent_model_outputs.csv")
save_latest_and_run(feature_importance, "risk_leadtime_feature_importance.csv")

joblib.dump(best_risk_model, MODEL_DIR / "risk_lgbm_pipeline.joblib")
joblib.dump(best_lead_model, MODEL_DIR / "leadtime_lgbm_pipeline.joblib")
joblib.dump(best_risk_model, OUTPUT_DIR / "risk_lgbm_pipeline.joblib")
joblib.dump(best_lead_model, OUTPUT_DIR / "leadtime_lgbm_pipeline.joblib")

metadata = {
    "run_id": RUN_ID,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "risk_model_version": "risk_lgbm_06_hsj_v1",
    "leadtime_model_version": "leadtime_lgbm_06_hsj_v1",
    "risk_selection_priority": ["validation_f1", "validation_false_positive_rate", "validation_precision", "validation_recall", "validation_average_precision", "validation_roc_auc"],
    "risk_threshold_strategy": risk_threshold_strategy,
    "risk_false_positive_guard": RISK_FP_GUARD,
    "risk_precision_guard": RISK_PRECISION_GUARD,
    "leadtime_selection_priority": ["validation_macro_f1", "validation_weighted_f1", "validation_macro_recall", "validation_accuracy"],
    "risk_feature_count": len(risk_features),
    "leadtime_feature_count": len(leadtime_features),
    "risk_features": risk_features,
    "leadtime_features": leadtime_features,
    "risk_best": best_risk,
    "risk_holdout": risk_holdout_metrics,
    "leadtime_best": best_lead,
    "leadtime_holdout": lead_holdout_metrics,
    "leadtime_buckets": {
        "short_0_24h": "0시간 초과 24시간 이하",
        "mid_24_72h": "24시간 초과 72시간 이하",
        "long_72h_plus": "72시간 초과",
    },
    "input_paths": {
        "trainable_windows": str(TRAINABLE_PATH),
        "feature_columns": str(FEATURE_COLUMNS_PATH),
        "anomaly_scores": str(ANOMALY_SCORES_PATH),
    },
}

for path in [OUTPUT_DIR / "risk_leadtime_model_metadata.json", RUN_DIR / "risk_leadtime_model_metadata.json"]:
    path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")

history_path = OUTPUT_DIR / "supervised_run_history.csv"
history_rows = []
for row in model_metrics.to_dict(orient="records"):
    compact = {"run_id": RUN_ID, "model": row.get("model"), "split": row.get("split"), "model_key": row.get("model_key")}
    for metric in ["accuracy", "precision", "recall", "f1", "false_positive_rate", "average_precision", "roc_auc", "macro_f1", "weighted_f1", "macro_precision", "macro_recall"]:
        compact[metric] = row.get(metric, np.nan)
    history_rows.append(compact)
history_update = pd.DataFrame(history_rows)
if history_path.exists():
    history = pd.concat([pd.read_csv(history_path), history_update], ignore_index=True)
else:
    history = history_update
history.to_csv(history_path, index=False)
history.to_csv(RUN_DIR / "supervised_run_history.csv", index=False)

print("saved to:", OUTPUT_DIR)
print("run dir:", RUN_DIR)

## 8. 실행 이력 시각화

반복 실행 시 run history를 통해 위험도와 리드타임 성능 추이를 확인한다.

In [ ]:
history = pd.read_csv(OUTPUT_DIR / "supervised_run_history.csv")
plot_history = history.copy()
plot_history["run_order"] = plot_history.groupby(["model", "split"]).cumcount() + 1

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

risk_plot = plot_history[(plot_history["model"].eq("risk_lgbm")) & (plot_history["split"].eq("validation"))]
if not risk_plot.empty:
    axes[0].plot(risk_plot["run_order"], risk_plot["f1"], marker="o", label="f1")
    axes[0].plot(risk_plot["run_order"], risk_plot["false_positive_rate"], marker="o", label="false_positive_rate")
    axes[0].set_title("Risk validation trend")
    axes[0].set_xlabel("run order")
    axes[0].legend()

lead_plot = plot_history[(plot_history["model"].eq("leadtime_lgbm")) & (plot_history["split"].eq("validation"))]
if not lead_plot.empty:
    axes[1].plot(lead_plot["run_order"], lead_plot["macro_f1"], marker="o", label="macro_f1")
    axes[1].plot(lead_plot["run_order"], lead_plot["weighted_f1"], marker="o", label="weighted_f1")
    axes[1].set_title("Leadtime validation trend")
    axes[1].set_xlabel("run order")
    axes[1].legend()

plt.tight_layout()
plot_path = OUTPUT_DIR / "supervised_run_history_plot.png"
run_plot_path = PLOT_DIR / "supervised_run_history_plot.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.savefig(run_plot_path, dpi=150, bbox_inches="tight")
plt.show()

display(history.tail(10))